In [1]:
import os
import time
import joblib
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score 
from imblearn.over_sampling import SMOTENC

In [2]:
def haversine_np(lon1, lat1, lon2, lat2):
    """
    Calculate the great circle distance between two points 
    on the earth (specified in decimal degrees) using Haversine formula.
    """
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat / 2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    km = 6367 * c
    return km

In [3]:
CATEGORY_DISPLAY_NAMES = {
    'grocery_pos': '🛒 In-Store Grocery & Supermarket',
    'shopping_net': '🌐 Online Shopping & E-Commerce',
    'shopping_pos': '🏬 In-Store Retail & Department Stores',
    'misc_net': '💻 Online Services & Digital Transfers',
    'misc_pos': '🏪 In-Store Miscellaneous & General Goods',
    'gas_transport': '⛽ Gas Station, Fuel & Transportation',
    'food_dining': '🍽️ Restaurant, Bars & Dining',
    'travel': '✈️ Travel, Airlines & Hotels',
    'entertainment': '🎟️ Entertainment, Movies & Concerts',
    'health_fitness': '🏥 Pharmacy, Healthcare & Gyms',
    'kids_pets': '🐾 Kids, Toys & Pet Supplies',
    'personal_care': '💄 Personal Care, Salons & Cosmetics',
    'home': '🏠 Home Improvement & Furnishings',
    'grocery_net': '📦 Online Grocery Delivery'
}

In [4]:
data_path = os.path.join(os.getcwd(), 'data', 'dataset.csv')

if not os.path.exists(data_path):
    raise FileNotFoundError(f"Dataset not found at {data_path}")

print("=" * 60)
print("AI FRAUD SENTINEL: TRAINING NEW INTUITIVE ML MODEL")
print("=" * 60)
print(f"Loading transaction records from {data_path}...")
    
t0 = time.time()
df = pd.read_csv(data_path, nrows=600000)
print(f"Loaded {len(df):,} transactions in {time.time() - t0:.1f}s.")
    
df['distance_km'] = haversine_np(df['long'], df['lat'], df['merch_long'], df['merch_lat'])

df['trans_date_trans_time'] = pd.to_datetime(df['trans_date_trans_time'])
df['hour'] = df['trans_date_trans_time'].dt.hour
df['day_of_week'] = df['trans_date_trans_time'].dt.dayofweek
df['is_weekend'] = (df['day_of_week'] >= 5).astype(int)

df['dob'] = pd.to_datetime(df['dob'])
df['age'] = ((df['trans_date_trans_time'] - df['dob']).dt.days // 365.25).astype(int)

df['is_online'] = df['category'].str.endswith('_net').astype(int)

AI FRAUD SENTINEL: TRAINING NEW INTUITIVE ML MODEL
Loading transaction records from d:\Fraud_detection\Frad_detection_complete\data\dataset.csv...
Loaded 600,000 transactions in 2.4s.


In [5]:
df.head()

,Unnamed: 0,trans_date_trans_time,cc_num,merchant,category,amt,first,last,gender,street,...,unix_time,merch_lat,merch_long,is_fraud,distance_km,hour,day_of_week,is_weekend,age,is_online
0,0,2019-01-01 00:00:18,2703186189652095,"fraud_Rippin, Kub and Mann",misc_net,4.97,Jennifer,Banks,F,561 Perry Cove,...,1325376018,36.011293,-82.048315,0,78.548221,0,1,0,30,1
1,1,2019-01-01 00:00:44,630423337322,"fraud_Heller, Gutmann and Zieme",grocery_pos,107.23,Stephanie,Gill,F,43039 Riley Greens Suite 393,...,1325376044,49.159047,-118.186462,0,30.193207,0,1,0,40,0
2,2,2019-01-01 00:00:51,38859492057661,fraud_Lind-Buckridge,entertainment,220.11,Edward,Sanchez,M,594 White Dale Suite 530,...,1325376051,43.150704,-112.154481,0,108.138146,0,1,0,56,0
3,3,2019-01-01 00:01:16,3534093764340240,"fraud_Kutch, Hermiston and Farrell",gas_transport,45.00,Jeremy,White,M,9443 Cynthia Court Apt. 038,...,1325376076,47.034331,-112.561071,0,95.613163,0,1,0,51,0
4,4,2019-01-01 00:03:06,375534208663984,fraud_Keeling-Crist,misc_pos,41.96,Tyler,Garcia,M,408 Bradley Rest,...,1325376186,38.674999,-78.632459,0,77.508050,0,1,0,32,0


In [6]:
# Category encoding
cats = sorted(df['category'].unique())
cat2idx = {c: i for i, c in enumerate(cats)}
df['category_idx'] = df['category'].map(cat2idx)

In [17]:
df['category'].unique()

<StringArray>
[      'misc_net',    'grocery_pos',  'entertainment',  'gas_transport',
       'misc_pos',    'grocery_net',   'shopping_net',   'shopping_pos',
    'food_dining',  'personal_care', 'health_fitness',         'travel',
      'kids_pets',           'home']
Length: 14, dtype: str

In [7]:
cat2idx

{'entertainment': 0,
 'food_dining': 1,
 'gas_transport': 2,
 'grocery_net': 3,
 'grocery_pos': 4,
 'health_fitness': 5,
 'home': 6,
 'kids_pets': 7,
 'misc_net': 8,
 'misc_pos': 9,
 'personal_care': 10,
 'shopping_net': 11,
 'shopping_pos': 12,
 'travel': 13}

In [8]:
# Final feature set (100% intuitive and actionable)
feature_cols = [
    'amt',            # Transaction Amount ($)
    'category_idx',   # Spending Category (grocery, shopping, travel, etc.)
    'hour',           # Hour of Day (0-23, detects high-risk late night windows)
    'day_of_week',    # Day of Week (0=Monday, 6=Sunday)
    'is_weekend',     # Weekend flag (0 or 1)
    'age',            # Cardholder Age
    'distance_km',    # Distance from Home to Merchant (km)
    'city_pop',       # Cardholder City/Area Population Size
    'is_online'       # Online / Card-Not-Present flag (0 or 1)
]

X = df[feature_cols]
y = df['is_fraud']

print(f"Total Features ({len(feature_cols)}): {feature_cols}")
print(f"Class Distribution: Legit={sum(y==0):,}, Fraud={sum(y==1):,} ({y.mean()*100:.2f}%)")

Total Features (9): ['amt', 'category_idx', 'hour', 'day_of_week', 'is_weekend', 'age', 'distance_km', 'city_pop', 'is_online']
Class Distribution: Legit=596,482, Fraud=3,518 (0.59%)


In [9]:
# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [10]:
f'{y_train.mean()*100:.1f}%'

'0.6%'

In [11]:
categorical_indices = [
    1,
    2,
    3,
    4,
    8
]

smaoth_nc = SMOTENC(
    categorical_features=categorical_indices,
    random_state=42,
    k_neighbors=5,
    sampling_strategy=0.1
)

In [12]:
X_train_resample, y_train_resample = (
    smaoth_nc.fit_resample(X_train, y_train)
)

In [13]:
f'{y_train_resample.mean()*100:.1f}%'

'9.1%'

In [14]:
X_train_resample.columns

Index(['amt', 'category_idx', 'hour', 'day_of_week', 'is_weekend', 'age',
       'distance_km', 'city_pop', 'is_online'],
      dtype='str')

In [15]:
X_train_resample.columns

Index(['amt', 'category_idx', 'hour', 'day_of_week', 'is_weekend', 'age',
       'distance_km', 'city_pop', 'is_online'],
      dtype='str')

In [28]:
X_train_resample['is_weekend'] = (
    X_train_resample['day_of_week'] >= 5
).astype(int)

idx_to_category = {
    idx: category for category, idx in cat2idx.items()
}

X_train_resample['is_online'] = (
    X_train_resample['category_idx'].map(idx_to_category).str.endswith('_net').astype(int)
)

In [29]:
t_time = time.time()

model = RandomForestClassifier(
    n_estimators=100,
    max_depth=16,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_resample, y_train_resample)

print(f"time for trianing model after resmpling - \n {time.time() - t_time:.2f}")

time for trianing model after resmpling - 
 34.77


In [30]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:,1]

# Matrics 

In [31]:

from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    average_precision_score,
    accuracy_score
)

# Predict on the original, untouched test data
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# 1. Accuracy
print("Accuracy:", accuracy_score(y_test, y_pred))

# 2. Confusion matrix
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

# 3. Precision, Recall, F1-score
print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["Legitimate", "Fraud"],
    zero_division=0
))

# 4. Precision-Recall AUC (Average Precision)
print("Average Precision (PR-AUC):",
      average_precision_score(y_test, y_prob))


Accuracy: 0.9976583333333333

Confusion Matrix:
[[119135    161]
 [   120    584]]

Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      1.00      1.00    119296
       Fraud       0.78      0.83      0.81       704

    accuracy                           1.00    120000
   macro avg       0.89      0.91      0.90    120000
weighted avg       1.00      1.00      1.00    120000

Average Precision (PR-AUC): 0.8707468284532828


In [32]:
# Evaluation
y_prob = model.predict_proba(X_test)[:, 1]
y_pred = (y_prob >= 0.5).astype(int)
auc = roc_auc_score(y_test, y_prob)

print("\n" + "=" * 60)
print("MODEL PERFORMANCE METRICS")
print("=" * 60)
print(f"ROC-AUC Score: {auc:.4f}")
print(classification_report(y_test, y_pred, target_names=['Legitimate (0)', 'Fraudulent (1)']))


MODEL PERFORMANCE METRICS
ROC-AUC Score: 0.9928
                precision    recall  f1-score   support

Legitimate (0)       1.00      1.00      1.00    119296
Fraudulent (1)       0.78      0.83      0.81       704

      accuracy                           1.00    120000
     macro avg       0.89      0.91      0.90    120000
  weighted avg       1.00      1.00      1.00    120000



In [34]:
# Feature Importance Breakdown
importances = pd.Series(model.feature_importances_, index=feature_cols).sort_values(ascending=False)
print("\nFeature Importances:")
for feat, imp in importances.items():
    print(f"  • {feat:15s}: {imp * 100:5.2f}%")


Feature Importances:
  • amt            : 56.72%
  • hour           : 18.37%
  • category_idx   : 12.29%
  • city_pop       :  3.82%
  • age            :  3.45%
  • is_online      :  2.86%
  • distance_km    :  1.39%
  • day_of_week    :  0.89%
  • is_weekend     :  0.21%


In [ ]:
# Save artifacts
print("\nSaving production model artifacts...")
artifacts = {
    'categories': cats,
    'cat2idx': cat2idx,
    'category_display_names': CATEGORY_DISPLAY_NAMES,
    'features': feature_cols,
    'feature_importances': importances.to_dict(),
    'roc_auc': float(auc)
}

joblib.dump(model, 'fraud_detection_model.pkl')
joblib.dump(artifacts, 'label_encoder.jb')
print("Done! Saved 'fraud_detection_model.pkl' and 'label_encoder.jb' successfully.")


# Camparing Model With and Without SMOTENC

In [ ]:

# ============================================================
# 1. TRAIN / TEST SPLIT
# ============================================================
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    average_precision_score,
    roc_auc_score
)
from imblearn.over_sampling import SMOTENC
from sklearn.ensemble import RandomForestClassifier
import pandas as pd
import numpy as np
import time
import joblib

print("=" * 70)
print("AI FRAUD SENTINEL: SMOTENC MODEL COMPARISON")
print("=" * 70)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("\nOriginal training class distribution:")
print(y_train.value_counts().sort_index())

print("\nUntouched test class distribution:")
print(y_test.value_counts().sort_index())


# ============================================================
# 2. MODEL A: WITHOUT SMOTENC
# ============================================================
print("\n" + "=" * 70)
print("MODEL A: RANDOM FOREST WITHOUT SMOTENC")
print("=" * 70)

start_time = time.time()

model_without_smotenc = RandomForestClassifier(
    n_estimators=100,
    max_depth=16,
    random_state=42,
    n_jobs=-1,
    class_weight=None
)

model_without_smotenc.fit(X_train, y_train)

time_without_smotenc = time.time() - start_time

print(
    f"Training time without SMOTENC: "
    f"{time_without_smotenc:.2f} seconds"
)


# ============================================================
# 3. MODEL B: WITH SMOTENC
# ============================================================
print("\n" + "=" * 70)
print("MODEL B: RANDOM FOREST WITH SMOTENC")
print("=" * 70)

# Feature positions in feature_cols:
# 0 = amt
# 1 = category_idx
# 2 = hour
# 3 = day_of_week
# 4 = is_weekend
# 5 = age
# 6 = distance_km
# 7 = city_pop
# 8 = is_online

categorical_indices = [1, 2, 3, 4, 8]

smotenc = SMOTENC(
    categorical_features=categorical_indices,
    random_state=42,
    k_neighbors=5,
    sampling_strategy=0.1
)

start_time = time.time()

# IMPORTANT: resample training data only
X_train_resampled, y_train_resampled = smotenc.fit_resample(
    X_train,
    y_train
)

# Restore the category index to integer form
X_train_resampled["category_idx"] = (
    X_train_resampled["category_idx"].round().astype(int)
)

# Recalculate features that are determined by other features
X_train_resampled["day_of_week"] = (
    X_train_resampled["day_of_week"].round().astype(int)
)

X_train_resampled["is_weekend"] = (
    X_train_resampled["day_of_week"] >= 5
).astype(int)

idx_to_category = {
    idx: category
    for category, idx in cat2idx.items()
}

category_names = X_train_resampled["category_idx"].map(
    idx_to_category
)

if category_names.isna().any():
    raise ValueError(
        "Unknown category index found after SMOTENC."
    )

X_train_resampled["is_online"] = (
    category_names.str.endswith("_net").astype(int)
)

time_resampling = time.time() - start_time

print("\nTraining distribution after SMOTENC:")
print(y_train_resampled.value_counts().sort_index())
print(f"Resampling time: {time_resampling:.2f} seconds")

start_time = time.time()

# Same Random Forest settings as Model A
model_with_smotenc = RandomForestClassifier(
    n_estimators=100,
    max_depth=16,
    random_state=42,
    n_jobs=-1,
    class_weight=None
)

model_with_smotenc.fit(
    X_train_resampled,
    y_train_resampled
)

time_with_smotenc = time.time() - start_time

print(
    f"Training time with SMOTENC: "
    f"{time_with_smotenc:.2f} seconds"
)


# ============================================================
# 4. COMMON EVALUATION FUNCTION
# ============================================================
def evaluate_model(model_name, trained_model, X_test, y_test):
    # Predict on the same untouched test set
    y_prob = trained_model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.5).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_test,
        y_pred,
        labels=[0, 1]
    ).ravel()

    metrics = {
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Fraud Precision": precision_score(
            y_test, y_pred, zero_division=0
        ),
        "Fraud Recall": recall_score(
            y_test, y_pred, zero_division=0
        ),
        "Fraud F1": f1_score(
            y_test, y_pred, zero_division=0
        ),
        "PR-AUC": average_precision_score(y_test, y_prob),
        "ROC-AUC": roc_auc_score(y_test, y_prob),
        "False Positives": fp,
        "Missed Fraud": fn,
        "Detected Fraud": tp
    }

    print("\n" + "-" * 70)
    print(f"RESULTS: {model_name}")
    print("-" * 70)

    print(f"Accuracy:        {metrics['Accuracy']:.4f}")
    print(f"Fraud Precision: {metrics['Fraud Precision']:.4f}")
    print(f"Fraud Recall:    {metrics['Fraud Recall']:.4f}")
    print(f"Fraud F1:        {metrics['Fraud F1']:.4f}")
    print(f"PR-AUC:          {metrics['PR-AUC']:.4f}")
    print(f"ROC-AUC:         {metrics['ROC-AUC']:.4f}")

    print(f"\nFalse positives: {fp:,}")
    print(f"Missed fraud:    {fn:,}")
    print(f"Detected fraud:  {tp:,}")

    print("\nConfusion Matrix:")
    print(
        confusion_matrix(
            y_test,
            y_pred,
            labels=[0, 1]
        )
    )

    print("\nClassification Report:")
    print(
        classification_report(
            y_test,
            y_pred,
            labels=[0, 1],
            target_names=["Legitimate", "Fraud"],
            zero_division=0
        )
    )

    return metrics


# ============================================================
# 5. EVALUATE BOTH MODELS
# ============================================================
results = []

results.append(
    evaluate_model(
        "Without SMOTENC",
        model_without_smotenc,
        X_test,
        y_test
    )
)

results.append(
    evaluate_model(
        "With SMOTENC",
        model_with_smotenc,
        X_test,
        y_test
    )
)


# ============================================================
# 6. SIDE-BY-SIDE COMPARISON
# ============================================================
comparison = pd.DataFrame(results).set_index("Model")

print("\n" + "=" * 90)
print("FINAL MODEL COMPARISON")
print("=" * 90)

print(comparison.round(4).to_string())

print("\nWhich model scores higher on each metric?")

for metric in [
    "Fraud Precision",
    "Fraud Recall",
    "Fraud F1",
    "PR-AUC",
    "ROC-AUC"
]:
    print(f"{metric}: {comparison[metric].idxmax()}")

print("\nTraining times:")
print(f"Without SMOTENC: {time_without_smotenc:.2f} seconds")
print(f"With SMOTENC:    {time_with_smotenc:.2f} seconds")

comparison.to_csv("smotenc_model_comparison.csv")
print("\nSaved comparison to smotenc_model_comparison.csv")


# ============================================================
# 7. FEATURE IMPORTANCE FOR BOTH MODELS
# ============================================================
importance_without = pd.Series(
    model_without_smotenc.feature_importances_,
    index=feature_cols,
    name="Without SMOTENC"
).sort_values(ascending=False)

importance_with = pd.Series(
    model_with_smotenc.feature_importances_,
    index=feature_cols,
    name="With SMOTENC"
).sort_values(ascending=False)

importance_comparison = pd.concat(
    [importance_without, importance_with],
    axis=1
).fillna(0)

print("\nFEATURE IMPORTANCE COMPARISON (%)")
print((importance_comparison * 100).round(2).to_string())

importance_comparison.to_csv(
    "smotenc_feature_importance_comparison.csv"
)

In [ ]:
# ============================================================
# 8. SAVE BOTH MODELS AND THEIR METADATA
# ============================================================
joblib.dump(
    model_without_smotenc,
    "fraud_model_without_smotenc.pkl"
)

joblib.dump(
    model_with_smotenc,
    "fraud_model_with_smotenc.pkl"
)

artifacts = {
    "categories": cats,
    "cat2idx": cat2idx,
    "category_display_names": CATEGORY_DISPLAY_NAMES,
    "features": feature_cols,
    "comparison_metrics": comparison.to_dict(),
    "feature_importances_without_smotenc":
        importance_without.to_dict(),
    "feature_importances_with_smotenc":
        importance_with.to_dict(),
    "smotenc_sampling_strategy": 0.1
}

joblib.dump(artifacts, "fraud_model_comparison_artifacts.pkl")

print("\nSaved both models and comparison artifacts successfully.")
